In [9]:
# PROJECT MINERVA - HIGHEST RATED TEST QUESTION
# All lower-rated test questions were solved successfully before this run.

# !pip install -q langgraph langchain-google-genai

import os
from typing import TypedDict, List

from google.colab import userdata
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2
)

class AgentState(TypedDict):
    problem: str
    analysis: str
    code: str
    review: str
    retries: int
    memory: List[str]
    test_cases: str

def analyze_problem(state: AgentState):

    prompt = f"""
You are a competitive programmer.

Analyze the following Codeforces problem.

Problem:
{state['problem']}

Provide:

1. Main idea
2. Algorithm
3. Time complexity
4. Important edge cases

Keep the answer concise.
"""

    response = llm.invoke(prompt)

    state["analysis"] = response.content

    return state


def generate_solution(state: AgentState):

    previous_mistakes = "\n".join(state["memory"])

    prompt = f"""
You are solving a Codeforces problem.

Problem:
{state['problem']}

Analysis:
{state['analysis']}

Previous mistakes:
{previous_mistakes}

Write a complete C++17 solution.

Output ONLY code.
"""

    response = llm.invoke(prompt)

    state["code"] = response.content

    return state


def generate_tests(state: AgentState):

    prompt = f"""
Problem:
{state['problem']}

Generate 3 important test cases for validating a solution.

Format:

1.
Input:
...

Expected Output:
...

2.
Input:
...

Expected Output:
...

3.
Input:
...

Expected Output:
...
"""

    response = llm.invoke(prompt)

    state["test_cases"] = response.content

    return state


def review_solution(state: AgentState):

    prompt = f"""
You are a competitive programming reviewer.

Problem:
{state['problem']}

Solution:
{state['code']}

Test Cases:
{state['test_cases']}

Check:

1. Logic correctness
2. Edge cases
3. Complexity

Reply ONLY in one of these formats:

PASS

or

FAIL: reason
"""

    response = llm.invoke(prompt)

    state["review"] = response.content.strip()

    return state


def store_failure(state: AgentState):

    state["memory"].append(state["review"])

    state["retries"] += 1

    return state


def route_after_review(state: AgentState):

    if state["review"].startswith("PASS"):
        return END

    if state["retries"] >= 3:
        return END

    return "store_failure"


graph = StateGraph(AgentState)

graph.add_node("analyze_problem", analyze_problem)
graph.add_node("generate_solution", generate_solution)
graph.add_node("generate_tests", generate_tests)
graph.add_node("review_solution", review_solution)
graph.add_node("store_failure", store_failure)

graph.set_entry_point("analyze_problem")

graph.add_edge("analyze_problem", "generate_solution")
graph.add_edge("generate_solution", "generate_tests")
graph.add_edge("generate_tests", "review_solution")

graph.add_conditional_edges(
    "review_solution",
    route_after_review
)

graph.add_edge(
    "store_failure",
    "generate_solution"
)

agent = graph.compile()

problem = """
There are n
 children in a class, m
 pairs among them are friends. The i
-th pair who are friends have a friendship value of fi
.

The teacher has to go for k
 excursions, and for each of the excursions she chooses a pair of children randomly, equiprobably and independently. If a pair of children who are friends is chosen, their friendship value increases by 1
 for all subsequent excursions (the teacher can choose a pair of children more than once). The friendship value of a pair who are not friends is considered 0
, and it does not change for subsequent excursions.

Find the expected value of the sum of friendship values of all k
 pairs chosen for the excursions (at the time of being chosen). It can be shown that this answer can always be expressed as a fraction pq
 where p
 and q
 are coprime integers. Calculate p⋅q−1mod(109+7)
.

Input
Each test contains multiple test cases. The first line contains the number of test cases t
 (1≤t≤5⋅104
). Description of the test cases follows.

The first line of each test case contains 3
 integers n
, m
 and k
 (2≤n≤105
, 0≤m≤min(105
, n(n−1)2)
, 1≤k≤2⋅105
) — the number of children, pairs of friends and excursions respectively.

The next m
 lines contain three integers each — ai
, bi
, fi
 — the indices of the pair of children who are friends and their friendship value. (ai≠bi
, 1≤ai,bi≤n
, 1≤fi≤109
). It is guaranteed that all pairs of friends are distinct.

It is guaranteed that the sum of n
 and sum m
 over all test cases does not exceed 105
 and the sum of k
 over all test cases does not exceed 2⋅105
.

Output
For each test case, print one integer — the answer to the problem.
"""

initial_state = {
    "problem": problem,
    "analysis": "",
    "code": "",
    "review": "",
    "retries": 0,
    "memory": [],
    "test_cases": ""
}

final_state = agent.invoke(initial_state)

print("\n==================================================")
print("PROJECT MINERVA - COMPETITIVE PROGRAMMER AGENT")
print("==================================================")

print("\nProblem Summary")
print("---------------")
print(final_state["problem"])

print("\nAnalysis")
print("--------")
print(final_state["analysis"])

print("\nTest Cases Used By Reviewer")
print("---------------------------")
print(final_state["test_cases"])

print("\nFinal Review")
print("------------")
print(final_state["review"])

print("\nRetries Used")
print("------------")
print(final_state["retries"])

print("\nMemory Entries")
print("--------------")
print(len(final_state["memory"]))

print("\nGenerated Solution")
print("------------------")
print(final_state["code"])

print("\n==================================================")
print("AGENT FINISHED SUCCESSFULLY")
print("==================================================")


PROJECT MINERVA - COMPETITIVE PROGRAMMER AGENT

Problem Summary
---------------

There are n
 children in a class, m
 pairs among them are friends. The i
-th pair who are friends have a friendship value of fi
.

The teacher has to go for k
 excursions, and for each of the excursions she chooses a pair of children randomly, equiprobably and independently. If a pair of children who are friends is chosen, their friendship value increases by 1
 for all subsequent excursions (the teacher can choose a pair of children more than once). The friendship value of a pair who are not friends is considered 0
, and it does not change for subsequent excursions.

Find the expected value of the sum of friendship values of all k
 pairs chosen for the excursions (at the time of being chosen). It can be shown that this answer can always be expressed as a fraction pq
 where p
 and q
 are coprime integers. Calculate p⋅q−1mod(109+7)
.

Input
Each test contains multiple test cases. The first line contains the